In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib, pandas as pd

SAIDA = pathlib.Path("/content/drive/MyDrive/precos-combustiveis-anp/dados_tratados")
df = pd.read_parquet(SAIDA / "precos_tratados.parquet")

# o cadastro de cada CNPJ é único?
chk = df.groupby("cnpj")[["uf", "municipio", "regiao"]].nunique()
print("CNPJs com mais de uma UF/município/região:", (chk > 1).any(axis=1).sum())

# dim_posto: um registro por CNPJ, com o cadastro mais recente
dim_posto = (df.sort_values("data_coleta")
               .drop_duplicates("cnpj", keep="last")
               [["cnpj", "revenda", "municipio", "uf", "regiao"]])
print("postos:", len(dim_posto), "| CNPJ únicos:", df["cnpj"].nunique())

dim_produto = pd.DataFrame({
    "produto": ["gasolina", "gasolina_aditivada", "etanol", "diesel_s10", "diesel"],
    "familia": ["gasolina", "gasolina", "etanol", "diesel", "diesel"],
})

fato = df[["cnpj", "produto", "data_coleta", "valor_venda",
           "bandeira", "preco_atipico"]].copy()
fato["data_coleta"] = fato["data_coleta"].dt.strftime("%Y-%m-%d")

dim_posto.to_csv(SAIDA / "dim_posto.csv", index=False)
dim_produto.to_csv(SAIDA / "dim_produto.csv", index=False)
fato.to_csv(SAIDA / "fato_preco.csv", index=False)
print("fato:", len(fato))
print(df["produto"].value_counts())   # guarde: vamos comparar com o banco

Mounted at /content/drive
CNPJs com mais de uma UF/município/região: 5
postos: 21799 | CNPJ únicos: 21799
fato: 2964388
produto
gasolina              788126
etanol                666357
gasolina_aditivada    613155
diesel_s10            556250
diesel                340500
Name: count, dtype: int64


In [2]:
cnpjs = chk[(chk > 1).any(axis=1)].index
(df[df["cnpj"].isin(cnpjs)]
   .groupby(["cnpj", "uf", "municipio"])["data_coleta"].agg(["min", "max", "count"]))

min        max  count
cnpj               uf municipio                                       
06.295.187/0001-84 RS ALVORADA            2024-07-03 2024-10-23     55
                      PORTO ALEGRE        2023-04-05 2024-06-19    136
08.947.513/0001-16 MA SAO JOSE DE RIBAMAR 2023-01-04 2024-06-25    109
                      SAO LUIS            2024-07-02 2026-05-05     81
09.164.791/0016-46 RS GRAVATAI            2024-07-01 2026-06-29    147
                      SAPUCAIA DO SUL     2024-04-17 2024-06-24     30
26.947.902/0001-23 RS GRAVATAI            2024-07-24 2025-06-25    112
                      PORTO ALEGRE        2026-05-13 2026-06-24     12
30.649.402/0001-82 RS NOVO HAMBURGO       2023-04-26 2025-06-25     19
                      PORTO ALEGRE        2026-06-02 2026-06-29      8